Termolab python


1.) Beregn massestrømmen av kuldemedia $\dot{m}, [\frac{kg}{s}]$, for hver målingsserie dere har registrert ved å bruke energibalansen til et åpent system:

$$
\dot{W}_{komp} = \dot{m}(h_{inn}-h_{ut})
$$
 
som gir oss
$$
\frac{\dot{W}_{komp}}{(h_{inn}-h_{ut})} = \dot{m}
$$
hvor 
$$
h_{ut} = h_{inn} + \frac{h_{2,is}-h_{inn}}{\eta_{is}}
$$
og 
$$
\eta_{is} = 0.0017\pi^3 - 0.0286\pi^2 + 0.1533\pi + 0.1642
$$
$$
\pi = \frac{p_{kondenser}}{p_{fordamper}}
$$

In [62]:
import pandas as pd
import numpy as np

# m trenger: W, h_inn, h_ut, 
# h_ut trenger: h_inn, h_2,is, eta_is
# eta_is trenger: p_kond, p_ford
# p_kond, p_ford trenger: t_kond, t_ford
# h_2,is trenger: 

df = pd.read_excel("merged log termo.xlsx", sheet_name="log")  

df.columns = ["time", "T1", "T2", "T3", "T4", "Thot", "Tcold", "W_komp", "Tkond", "Tford"]   
            #hh:mm:ss, degC,, degC, degC, degC, degC, degC,      W,       degC,    degC 
# finner W
W_komp = df["W_komp"].to_numpy()


In [63]:
from CoolProp.CoolProp import PropsSI
#finner entalpier


fluid = "R134a"
kelvin = 273.15
manual_tid= ["12:24:01", "12:25:59", "12:28:08", "12:30:06", "12:32:04"]

log_date = df["time"].dt.date.iloc[0]

start_time, end_time = pd.to_datetime([
    f"{log_date} {manual_tid[0]}",
    f"{log_date} {manual_tid[-1]}"])

manual_data = df.loc[df["time"].between(start_time, end_time)].copy()

#planen var å bruke  en fuknsjon til å fikse alle  data, men det er for stor variasjon i demands
def entalpi(data, Ta, Tb):
    
    """
    input: data -> dataset
    Ta er første temperatur, Tb er andre temperatur
    
    1. lager kopi av dataset
    2. finner fordampnings- og konsenseringstrykk
    3. bruker trykkene for å finne den isentropiske virkningsgraden
    4. finenr h1, h2, h2is ved hjelp av dataene i forrige steg
    5. returnerer dataset med original data + nye entalpier
    """

    data = data.copy()
    T_suc  = data[Ta].to_numpy() + kelvin   # [K]
    p_ford = PropsSI("P", "T", data["Tford"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]
    p_kond = PropsSI("P", "T", data["Tkond"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]

    data["superheat_K"] = data[Ta] - data[Tb]            
    data["pi"] = p_kond / p_ford
    pi = data["pi"]
    data["eta_is"] = 0.0017*pi**3 - 0.0286*pi**2 + 0.1533*pi + 0.1642

    h1   = PropsSI("H", "T", T_suc, "P", p_ford, fluid)       # [J/kg]
    s1   = PropsSI("S", "T", T_suc, "P", p_ford, fluid)       # [J/kgK]
    h2is = PropsSI("H", "P", p_kond, "S", s1, fluid)          # isentropisk 
    h2   = h1 + (h2is - h1) / data["eta_is"].to_numpy()       # ekte

    data["h1"], data["h2is"], data["h2"] = h1, h2is, h2       # [J/kg]
    return data

resultat_kond = entalpi(manual_data, "T1", "Tkond")
res_manual = resultat_kond[resultat_kond["time"].dt.strftime("%H:%M:%S").isin(manual_tid)]
print(res_manual[["time", "eta_is", "h1", "h2", "h2is"]])

                  time    eta_is             h1             h2           h2is
27 2026-10-06 12:24:01  0.396674  423448.046318  477115.550893  444736.562076
38 2026-10-06 12:25:59  0.411898  427709.421687  489917.418805  453332.749961
50 2026-10-06 12:28:08  0.421804  432295.215675  502613.469525  461955.740381
61 2026-10-06 12:30:06  0.428598  435674.298241  515587.507109  469924.934773
72 2026-10-06 12:32:04  0.429792  438680.891418  524540.959219  475582.827625


In [64]:
resultat_kond["m_dot"] = resultat_kond["W_komp"] / (resultat_kond["h2"] - resultat_kond["h1"])     # [kg/s]
resultat_kond["m_dot_e3"] = resultat_kond["m_dot"] * 1e3                        # [1e-3 kg/s]

res_manual = resultat_kond[resultat_kond["time"].dt.strftime("%H:%M:%S").isin(manual_tid)]

print(res_manual["m_dot"])

27    0.002739
38    0.002395
50    0.002147
61    0.001915
72    0.001817
Name: m_dot, dtype: float64


2. Beregne varmeytelse av kondensator i $Q_{kond}$, [W], for hver målingsserie dere har tatt ved å
bruke energibalansen til et åpent system:


$$
\dot{Q}_{kond} = \dot{m}(h_{ut}-h_{inn})
$$

$
h_{inn} – \text{ entalpien inn på kondensator}, [\frac{J}{kg}]$
, bruk T2 temperaturen og kondenseringstrykk $p_{kond}$.

$h_{ut} - \text{ entalpien ut av kondensator} , [\frac{J}{kg}]$
,bruk kondenseringstrykk, $p_{kond}$, og T3 temperaturen.

For å kompensere for varmetap mellom rør og omgivelsestemperaturen til kondensatoren, så
anbefales det at dere legger til 4.8 K til temperaturverdiene I T3 og T2.
Altså at dere tar:
Temperatur T3 = T3 + 4.8 K (VIKTIG: koeffisienten brukes ikke hvis T3 + 4.8 K> metnings
temperatur).
Temperatur T2 = T2 + 4.8 K

In [65]:
def entalpi2(data, Ta="T2", Tb="T3"):
 
    korreksjon_k = 4.8
    
    data = data.copy()
    p_kond = PropsSI("P", "T", data["Tkond"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]
    p_ford = PropsSI("P", "T", data["Tford"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]
    T_suc  = data[Ta].to_numpy() + kelvin + korreksjon_K  # [K]

    data["superheat_K"] = data[Ta] - (data[Tb]+korreksjon_K)            
    data["pi"] = p_kond / p_ford
    pi = data["pi"]
    data["eta_is"] = 0.0017*pi**3 - 0.0286*pi**2 + 0.1533*pi + 0.1642

    hinn   = PropsSI("H", "T", T_suc, "P", p_ford, fluid)       # [J/kg]
    s1   = PropsSI("S", "T", T_suc, "P", p_ford, fluid)       # [J/kgK]
    hut_is = PropsSI("H", "P", p_kond, "S", s1, fluid)          # isentropisk 
    hut   = hinn + (hut_is - hinn) / data["eta_is"].to_numpy()       # ekte

    data["hinn_ford"], data["hut_is_ford"], data["hut_ford"] = hinn, hut_is, hut    # [J/kg]
    return data


resultat_ford = entalpi2(manual_data, Ta="T2", Tb="T3")
res_manual_ford = resultat_kond[resultat_kond["time"].dt.strftime("%H:%M:%S").isin(manual_tid)]

res_manual_ford["Qdot_kond"] = res_manual_ford["m_dot"] * (res_manual_ford["hut_ford"] - res_manual_ford["hinn_ford"])     # [kg/s]

print(res_manual_ford[["time", "Qdot_kond"]])

KeyError: 'hut_ford'